In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


"""
Standardize Bronze reference snapshots into typed Silver reference tables
and isolate records that violate approved key, attribute, or hierarchy
rules.
"""

CATALOG = "semiconplus_portfolio"

BRONZE_TABLES = {
    "product_groups": f"{CATALOG}.bronze.ref_product_groups",
    "devices": f"{CATALOG}.bronze.ref_devices",
    "sites": f"{CATALOG}.bronze.ref_sites",
    "equipment": f"{CATALOG}.bronze.ref_equipment",
}

SILVER_TABLES = {
    name: f"{CATALOG}.silver.{name}"
    for name in BRONZE_TABLES
}

QUARANTINE_TABLE = f"{CATALOG}.quarantine.reference_records"
QUALITY_TABLE = f"{CATALOG}.monitoring.data_quality_results"
PRODUCTION_LOTS_TABLE = f"{CATALOG}.silver.production_lots"

EXPECTED_COUNTS = {
    "product_groups": 6,
    "devices": 30,
    "sites": 3,
    "equipment": 24,
}

PIPELINE_RUN_ID = str(uuid4())
PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — DEPENDENCY CHECKS
# ===================================================   

"""
Confirm that all Bronze reference snapshots and the validated Silver
production-lot table are available before transformation.
"""

required_tables = list(BRONZE_TABLES.values()) + [PRODUCTION_LOTS_TABLE]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

assert not missing_tables, f"Required tables are missing: {missing_tables}"

for entity_name, table_name in BRONZE_TABLES.items():
    actual_count = spark.table(table_name).count()
    expected_count = EXPECTED_COUNTS[entity_name]

    assert actual_count == expected_count, (
        f"{table_name}: expected {expected_count} records, "
        f"found {actual_count}."
    )

print("Silver reference dependencies passed.")

In [0]:
# ===================================================
# BLOCK 3 — STANDARDIZE PRODUCT GROUPS
# ===================================================   

"""
Standardize product-group identifiers and descriptive attributes, and
convert the source restriction indicator to a Boolean security attribute.
"""

product_groups_checked_df = (
    spark.table(BRONZE_TABLES["product_groups"])
    .select(
        F.upper(F.trim("product_group_id")).alias("product_group_id"),
        F.trim("product_group_name").alias("product_group_name"),
        F.trim("business_unit").alias("business_unit"),
        F.expr("try_cast(trim(restricted) AS boolean)").alias("restricted"),
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("product_group_id").isNull()
                | (F.col("product_group_id") == ""),
                F.lit("MISSING_PRODUCT_GROUP_ID"),
            ),
            F.when(
                F.col("product_group_name").isNull()
                | (F.col("product_group_name") == ""),
                F.lit("MISSING_PRODUCT_GROUP_NAME"),
            ),
            F.when(
                F.col("business_unit").isNull()
                | (F.col("business_unit") == ""),
                F.lit("MISSING_BUSINESS_UNIT"),
            ),
            F.when(
                F.col("restricted").isNull(),
                F.lit("INVALID_RESTRICTED_FLAG"),
            ),
        )),
    )
)

In [0]:

# ===================================================
# BLOCK 4 — STANDARDIZE SITES
# ===================================================  

"""
Standardize manufacturing-site attributes and validate that each site
contains the timezone required for later UTC/local-time reporting.
"""

sites_checked_df = (
    spark.table(BRONZE_TABLES["sites"])
    .select(
        F.upper(F.trim("site_id")).alias("site_id"),
        F.trim("site_name").alias("site_name"),
        F.trim("country").alias("country"),
        F.trim("timezone").alias("timezone"),
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("site_id").isNull() | (F.col("site_id") == ""),
                F.lit("MISSING_SITE_ID"),
            ),
            F.when(
                F.col("site_name").isNull() | (F.col("site_name") == ""),
                F.lit("MISSING_SITE_NAME"),
            ),
            F.when(
                F.col("country").isNull() | (F.col("country") == ""),
                F.lit("MISSING_COUNTRY"),
            ),
            F.when(
                F.col("timezone").isNull() | (F.col("timezone") == ""),
                F.lit("MISSING_TIMEZONE"),
            ),
        )),
    )
)

In [0]:
# ===================================================
# BLOCK 5 — STANDARDIZE DEVICES
# ===================================================  

"""
Standardize device identifiers and classifications, convert target yield
to its analytical type, and validate the product-group relationship.
"""

valid_product_group_ids_df = (
    product_groups_checked_df
    .filter(F.size("_quality_reasons") == 0)
    .select("product_group_id")
)

devices_base_df = (
    spark.table(BRONZE_TABLES["devices"])
    .select(
        F.upper(F.trim("device_id")).alias("device_id"),
        F.trim("device_name").alias("device_name"),
        F.upper(F.trim("product_group_id")).alias("product_group_id"),
        F.upper(F.trim("package_type")).alias("package_type"),
        F.expr("try_cast(trim(target_yield) AS decimal(9,6))").alias(
            "target_yield"
        ),
        F.upper(F.trim("lifecycle_status")).alias("lifecycle_status"),
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
)

devices_checked_df = (
    devices_base_df
    .join(
        F.broadcast(
            valid_product_group_ids_df.withColumn(
                "_product_group_valid",
                F.lit(True),
            )
        ),
        "product_group_id",
        "left",
    )
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("device_id").isNull() | (F.col("device_id") == ""),
                F.lit("MISSING_DEVICE_ID"),
            ),
            F.when(
                F.col("device_name").isNull() | (F.col("device_name") == ""),
                F.lit("MISSING_DEVICE_NAME"),
            ),
            F.when(
                F.col("product_group_id").isNull()
                | (F.col("product_group_id") == ""),
                F.lit("MISSING_PRODUCT_GROUP_ID"),
            ),
            F.when(
                F.col("_product_group_valid").isNull(),
                F.lit("UNKNOWN_PRODUCT_GROUP_ID"),
            ),
            F.when(
                F.col("package_type").isNull() | (F.col("package_type") == ""),
                F.lit("MISSING_PACKAGE_TYPE"),
            ),
            F.when(
                F.col("target_yield").isNull()
                | (F.col("target_yield") < F.lit(0))
                | (F.col("target_yield") > F.lit(1)),
                F.lit("INVALID_TARGET_YIELD"),
            ),
            F.when(
                ~F.col("lifecycle_status").isin(
                    "NPI",
                    "RAMP",
                    "MASS_PRODUCTION",
                    "MATURE",
                    "END_OF_LIFE",
                ),
                F.lit("INVALID_LIFECYCLE_STATUS"),
            ),
        )),
    )
    .drop("_product_group_valid")
)

In [0]:
# ===================================================
# BLOCK 6 — STANDARDIZE EQUIPMENT
# ===================================================  

"""
Standardize equipment attributes, convert rated throughput to an integer,
and validate the equipment-to-site relationship.
"""

valid_site_ids_df = (
    sites_checked_df
    .filter(F.size("_quality_reasons") == 0)
    .select("site_id")
)

equipment_base_df = (
    spark.table(BRONZE_TABLES["equipment"])
    .select(
        F.upper(F.trim("equipment_id")).alias("equipment_id"),
        F.upper(F.trim("site_id")).alias("site_id"),
        F.upper(F.trim("equipment_model")).alias("equipment_model"),
        F.upper(F.trim("equipment_type")).alias("equipment_type"),
        F.expr("try_cast(trim(rated_units_per_hour) AS int)").alias(
            "rated_units_per_hour"
        ),
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
)

equipment_checked_df = (
    equipment_base_df
    .join(
        F.broadcast(
            valid_site_ids_df.withColumn("_site_valid", F.lit(True))
        ),
        "site_id",
        "left",
    )
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("equipment_id").isNull()
                | (F.col("equipment_id") == ""),
                F.lit("MISSING_EQUIPMENT_ID"),
            ),
            F.when(
                F.col("site_id").isNull() | (F.col("site_id") == ""),
                F.lit("MISSING_SITE_ID"),
            ),
            F.when(
                F.col("_site_valid").isNull(),
                F.lit("UNKNOWN_SITE_ID"),
            ),
            F.when(
                F.col("equipment_model").isNull()
                | (F.col("equipment_model") == ""),
                F.lit("MISSING_EQUIPMENT_MODEL"),
            ),
            F.when(
                F.col("equipment_type").isNull()
                | (F.col("equipment_type") == ""),
                F.lit("MISSING_EQUIPMENT_TYPE"),
            ),
            F.when(
                F.col("rated_units_per_hour").isNull()
                | (F.col("rated_units_per_hour") <= 0),
                F.lit("INVALID_RATED_UNITS_PER_HOUR"),
            ),
        )),
    )
    .drop("_site_valid")
)


In [0]:
# ===================================================
# BLOCK 7 — CHECK BUSINESS-KEY UNIQUENESS
# ===================================================  

"""
Attach duplicate-key quality reasons before splitting accepted and
rejected reference records.
"""

checked_datasets = {
    "product_groups": (product_groups_checked_df, "product_group_id"),
    "devices": (devices_checked_df, "device_id"),
    "sites": (sites_checked_df, "site_id"),
    "equipment": (equipment_checked_df, "equipment_id"),
}

for entity_name, (entity_df, key_column) in list(checked_datasets.items()):
    duplicate_keys_df = (
        entity_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .select(key_column)
        .withColumn("_duplicate_key", F.lit(True))
    )

    checked_datasets[entity_name] = (
        entity_df
        .join(duplicate_keys_df, key_column, "left")
        .withColumn(
            "_quality_reasons",
            F.when(
                F.col("_duplicate_key") == True,
                F.array_union(
                    F.col("_quality_reasons"),
                    F.array(F.lit("DUPLICATE_BUSINESS_KEY")),
                ),
            ).otherwise(F.col("_quality_reasons")),
        )
        .drop("_duplicate_key"),
        key_column,
    )

print("Reference business-key checks completed.")

In [0]:
# ===================================================
# BLOCK 8 — SPLIT ACCEPTED AND QUARANTINED RECORDS
# ===================================================  

"""
Separate valid reference records from rejected records while preserving
quality reasons, source lineage, and pipeline-run metadata.
"""

accepted_datasets = {}
quarantine_datasets = []
quality_metrics = []

for entity_name, (entity_df, key_column) in checked_datasets.items():
    accepted_df = (
        entity_df
        .filter(F.size("_quality_reasons") == 0)
        .drop("_quality_reasons")
        .withColumn(
            "_silver_pipeline_run_id",
            F.lit(PIPELINE_RUN_ID),
        )
        .withColumn(
            "_silver_processed_at_utc",
            F.current_timestamp(),
        )
    )

    rejected_df = (
        entity_df
        .filter(F.size("_quality_reasons") > 0)
        .select(
            F.lit(entity_name).alias("source_entity"),
            F.col(key_column).cast("string").alias("business_key"),
            F.to_json(F.struct(*entity_df.columns)).alias("record_payload"),
            "_quality_reasons",
            "_source_file_path",
            "_source_file_name",
            "_source_file_modification_time",
            "_ingested_at_utc",
            "_bronze_pipeline_run_id",
            F.lit(PIPELINE_RUN_ID).alias("_silver_pipeline_run_id"),
            F.current_timestamp().alias("_quarantined_at_utc"),
        )
    )

    source_count = entity_df.count()
    accepted_count = accepted_df.count()
    rejected_count = rejected_df.count()

    assert accepted_count + rejected_count == source_count

    accepted_datasets[entity_name] = accepted_df
    quarantine_datasets.append(rejected_df)
    quality_metrics.append(
        (entity_name, source_count, accepted_count, rejected_count)
    )

display(
    spark.createDataFrame(
        quality_metrics,
        ["entity", "source_rows", "accepted_rows", "rejected_rows"],
    ).orderBy("entity")
)

In [0]:
# ===================================================
# BLOCK 9 — ENFORCE INITIAL REFERENCE BASELINE
# ===================================================  

"""
Confirm that the controlled initial reference dataset contains no rejected
records and that all expected business entities remain available.
"""

for entity_name, expected_count in EXPECTED_COUNTS.items():
    actual_count = accepted_datasets[entity_name].count()
    assert actual_count == expected_count, (
        f"{entity_name}: expected {expected_count} accepted records, "
        f"found {actual_count}."
    )

total_rejected_count = sum(
    rejected_df.count()
    for rejected_df in quarantine_datasets
)

assert total_rejected_count == 0, (
    f"Expected no invalid reference records, found {total_rejected_count}."
)

print("Initial Silver reference baseline passed.")

In [0]:
# ===================================================
# BLOCK 10 — WRITE SILVER REFERENCE TABLES
# ===================================================  

"""
Replace each Silver reference snapshot with its current validated source
population. Overwrite mode prevents duplicate records during reruns.
"""

for entity_name, accepted_df in accepted_datasets.items():
    target_table = SILVER_TABLES[entity_name]

    (
        accepted_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    persisted_count = spark.table(target_table).count()
    assert persisted_count == EXPECTED_COUNTS[entity_name]

    print(f"Written {target_table}: {persisted_count} records")

In [0]:
# ===================================================
# BLOCK 11 — WRITE REFERENCE QUARANTINE
# ===================================================  

"""
Persist the current rejected-reference snapshot even when it is empty so
future invalid reference deliveries have an established audit contract.
"""

combined_quarantine_df = quarantine_datasets[0]

for rejected_df in quarantine_datasets[1:]:
    combined_quarantine_df = combined_quarantine_df.unionByName(
        rejected_df,
        allowMissingColumns=True,
    )

(
    combined_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

assert spark.table(QUARANTINE_TABLE).count() == 0

print("Reference quarantine table written.")

In [0]:
# ===================================================
# BLOCK 12 — REVALIDATE SILVER PRODUCTION REFERENCES
# ===================================================  
"""
Confirm that every accepted Silver production lot resolves to approved
product-group, device, site, and equipment references.

Equipment is also checked against the lot's site to prevent valid keys
from being combined under an invalid site-equipment relationship.
"""

lots_df = spark.table(PRODUCTION_LOTS_TABLE).alias("lots")

devices_df = spark.table(SILVER_TABLES["devices"]).select(
    "device_id", "product_group_id"
).alias("devices")

sites_df = spark.table(SILVER_TABLES["sites"]).select(
    "site_id"
).alias("sites")

equipment_df = spark.table(SILVER_TABLES["equipment"]).select(
    "equipment_id", "site_id"
).alias("equipment")

lot_reference_failures_df = (
    lots_df
    .join(
        F.broadcast(devices_df),
        F.col("lots.device_id") == F.col("devices.device_id"),
        "left",
    )
    .join(
        F.broadcast(sites_df),
        F.col("lots.site_id") == F.col("sites.site_id"),
        "left",
    )
    .join(
        F.broadcast(equipment_df),
        (F.col("lots.equipment_id") == F.col("equipment.equipment_id"))
        & (F.col("lots.site_id") == F.col("equipment.site_id")),
        "left",
    )
    .filter(
        F.col("devices.device_id").isNull()
        | F.col("sites.site_id").isNull()
        | F.col("equipment.equipment_id").isNull()
        | (
            F.col("lots.product_group_id")
            != F.col("devices.product_group_id")
        )
    )
    .select("lots.*")
)

lot_reference_failure_count = lot_reference_failures_df.count()

display(lot_reference_failures_df.limit(20))

assert lot_reference_failure_count == 0, (
    f"Silver production lots contain {lot_reference_failure_count} "
    "unresolved or inconsistent reference relationships."
)

print("Silver production reference validation passed.")

In [0]:
# ===================================================
# BLOCK 13 — WRITE DATA-QUALITY RESULTS
# ===================================================  

"""
Append entity-level quality metrics for operational monitoring and
historical validation evidence.
"""

completed_at_utc = datetime.now(timezone.utc)

quality_rows = [
    (
        PIPELINE_RUN_ID,
        entity_name,
        source_rows,
        accepted_rows,
        rejected_rows,
        "PASSED" if rejected_rows == 0 else "FAILED",
        completed_at_utc,
    )
    for entity_name, source_rows, accepted_rows, rejected_rows
    in quality_metrics
]

quality_rows.append(
    (
        PIPELINE_RUN_ID,
        "production_lot_reference_integrity",
        lots_df.count(),
        lots_df.count() - lot_reference_failure_count,
        lot_reference_failure_count,
        "PASSED" if lot_reference_failure_count == 0 else "FAILED",
        completed_at_utc,
    )
)

quality_results_df = spark.createDataFrame(
    quality_rows,
    """
    pipeline_run_id STRING,
    dataset_name STRING,
    source_row_count LONG,
    accepted_row_count LONG,
    rejected_row_count LONG,
    validation_status STRING,
    validated_at_utc TIMESTAMP
    """,
)

(
    quality_results_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(QUALITY_TABLE)
)

display(quality_results_df.orderBy("dataset_name"))

In [0]:
# ===================================================
# BLOCK 14 — FINAL PIPELINE RESULT
# ===================================================  

"""
Publish the completed Silver reference transformation result for
execution logs and project evidence.
"""

PIPELINE_END_TIME = datetime.now(timezone.utc)

print("SILVER REFERENCE PIPELINE PASSED")
print("Product groups: 6")
print("Devices: 30")
print("Sites: 3")
print("Equipment: 24")
print("Reference quarantine records: 0")
print("Production-lot reference failures: 0")
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")
print(
    "Duration seconds: "
    f"{(PIPELINE_END_TIME - PIPELINE_START_TIME).total_seconds():.2f}" 
)